In [ ]:
import pandas as pd
import os
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
source_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
local_file = "clean_positive_6_fixed.csv"

print(f"reading clean_positive_6...")
df = pd.read_csv(source_path)
df['index_date'] = pd.to_datetime(df['index_date'], format='ISO8601', errors='coerce').dt.date
df = df.sort_values(by=['person_id', 'index_date'], ascending=True)
df_unique = df.drop_duplicates(subset=['person_id'], keep='first').copy()
df_unique['timeframe_start'] = (pd.to_datetime(df_unique['index_date']) - pd.DateOffset(months=6)).dt.date

df_final = df_unique[['person_id', 'index_date', 'timeframe_start', 'IsPositive']]

print(f"update to Bucket...")
df_final.to_csv(local_file, index=False)

subprocess.run(["gsutil", "cp", local_file, source_path], check=True)

if os.path.exists(local_file):
    os.remove(local_file)

print(f"done")
print(f"patients number：{len(df_final)}")

display(df_final.head(5))

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/

In [ ]:
!gsutil rm gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/clean_data/clean_positive.csv

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/

In [ ]:
!rm -rf ~/.gsutil/tracker-files/

In [ ]:
import pandas as pd
import os
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/conditions.csv"
target_csv = f"{bucket}/amia/clean_data/clean_condition_6.csv"
local_temp = "clean_condition_lean.csv"

anchor_df = pd.read_csv(anchor_path)
anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start']).dt.date
anchor_df['end'] = pd.to_datetime(anchor_df['index_date']).dt.date
window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False
drop_cols = ['condition_end_datetime', 'condition_type_concept_name', 'visit_occurrence_concept_name']


for chunk in pd.read_csv(source_csv, chunksize=chunk_size):
    chunk.drop(columns=drop_cols, errors='ignore', inplace=True)
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty: continue
    chunk['temp_dt'] = pd.to_datetime(chunk['condition_start_datetime'], format='ISO8601', utc=True).dt.date
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    
    in_window = (chunk['temp_dt'] >= chunk['win_start']) & (chunk['temp_dt'] < chunk['win_end'])
    chunk.loc[~in_window, 'condition_start_datetime'] = pd.NA
    
    chunk.drop(columns=['temp_dt', 'win_start', 'win_end'], inplace=True)
    
    mode = 'w' if not header_written else 'a'
    chunk.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)
    
!gsutil ls -lh {target_csv}

In [ ]:
import pandas as pd

condition_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_condition_6.csv"

df_stats = pd.read_csv(condition_path, usecols=['condition_start_datetime'])
total_rows = len(df_stats)
na_count = df_stats['condition_start_datetime'].isna().sum()

na_percentage = (na_count / total_rows) * 100

print(f"statistics：")
print(f"total rows {total_rows:,}")
print(f"time NA rows: {na_count:,}")
print(f"data out of the timeframe: {na_percentage:.2f}%")

In [ ]:
import pandas as pd
import os
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/measurements.csv"
target_csv = f"{bucket}/amia/clean_data/clean_measurement_6.csv"
local_temp = "clean_measurement_lean.csv"

print("loading")
anchor_df = pd.read_csv(anchor_path)

if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)

anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False
unique_concepts = set()

cols_to_keep = [
    'person_id', 'standard_concept_name', 'standard_concept_code', 
    'standard_vocabulary', 'measurement_datetime', 
    'value_as_number', 'value_as_concept_id','value_as_concept_name','unit_concept_id', 'unit_concept_name'
]

print(f"---measurement---")

for chunk in pd.read_csv(source_csv, usecols=cols_to_keep, chunksize=chunk_size):
    
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
        
    chunk['measurement_datetime'] = pd.to_datetime(chunk['measurement_datetime'], utc=True).dt.tz_localize(None)
    
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    
    in_window = (chunk['measurement_datetime'] >= chunk['win_start']) & \
                (chunk['measurement_datetime'] < chunk['win_end'])
    
    chunk.loc[~in_window, 'measurement_datetime'] = pd.NaT
    
    unique_concepts.update(chunk['standard_concept_name'].dropna().unique())
    
    chunk_to_save = chunk[cols_to_keep]
    
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

print(f"update to GCS")
subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)

print(f"standard_concept_name have {len(unique_concepts)} unique types")

print("\n[clean_measurement_6.head()] preview content：")
final_preview = pd.read_csv(target_csv, nrows=5)
print(final_preview.head())

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/

In [ ]:
import pandas as pd
import os
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/labs.csv"
target_csv = f"{bucket}/amia/clean_data/clean_lab_6.csv"
local_temp = "clean_lab_lean.csv"

anchor_df = pd.read_csv(anchor_path)

if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)

anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], format='ISO8601', utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], format='ISO8601', utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False
unique_concepts = set()

cols_to_keep = [
    'person_id', 'standard_concept_name', 'standard_concept_code', 
    'standard_vocabulary', 'measurement_datetime', 'value_as_number', 
    'value_as_concept_id', 'value_as_concept_name', 'unit_concept_id', 
    'unit_concept_name', 'range_low', 'range_high'
]


for chunk in pd.read_csv(source_csv, usecols=cols_to_keep, chunksize=chunk_size):
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
    chunk['measurement_datetime'] = pd.to_datetime(chunk['measurement_datetime'], format='ISO8601', utc=True).dt.tz_localize(None)
    
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    
    in_window = (chunk['measurement_datetime'] >= chunk['win_start']) & \
                (chunk['measurement_datetime'] < chunk['win_end'])
    
    chunk.loc[~in_window, 'measurement_datetime'] = pd.NaT
    
    chunk['measurement_datetime'] = chunk['measurement_datetime'].dt.strftime('%Y-%m-%d')

    unique_concepts.update(chunk['standard_concept_name'].dropna().unique())
    
    chunk_to_save = chunk[cols_to_keep]
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)

print(f"standard_concept_name has {len(unique_concepts)} unique lab types。")
final_preview = pd.read_csv(target_csv, nrows=5)
print(final_preview.head())

In [ ]:
import pandas as pd
import os
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/procedures.csv"  
target_csv = f"{bucket}/amia/clean_data/clean_procedure_6.csv"
local_temp = "clean_procedure_lean.csv"

anchor_df = pd.read_csv(anchor_path)

if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)

anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], format='ISO8601', utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], format='ISO8601', utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False


for chunk in pd.read_csv(source_csv, chunksize=chunk_size):
    if 'visit_occurrence_concept_name' in chunk.columns:
        chunk.drop(columns=['visit_occurrence_concept_name'], inplace=True)
    
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
    
    chunk['procedure_datetime'] = pd.to_datetime(chunk['procedure_datetime'], format='ISO8601', utc=True).dt.tz_localize(None)
    
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    
    in_window = (chunk['procedure_datetime'] >= chunk['win_start']) & \
                (chunk['procedure_datetime'] < chunk['win_end'])
    
    chunk.loc[~in_window, 'procedure_datetime'] = pd.NaT
    
    chunk['procedure_datetime'] = chunk['procedure_datetime'].dt.strftime('%Y-%m-%d')
    
    cols_to_save = [c for c in chunk.columns if c not in ['win_start', 'win_end']]
    chunk_to_save = chunk[cols_to_save]
    
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)

print(f"saved to: {target_csv}")

print("\n[clean_procedure_6.head()] preview：")
final_preview = pd.read_csv(target_csv, nrows=5)
print(final_preview.head())

In [ ]:
import pandas as pd

condition_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_procedure_6.csv"

df_stats = pd.read_csv(condition_path, usecols=['procedure_datetime'])

total_rows = len(df_stats)

na_count = df_stats['procedure_datetime'].isna().sum()

na_percentage = (na_count / total_rows) * 100


print(f"total rows: {total_rows:,}")
print(f"NA time rows: {na_count:,}")
print(f"data outside the timewindow: {na_percentage:.2f}%")


In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/clean_data/

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display 

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/drugs.csv"
target_csv = f"{bucket}/amia/clean_data/clean_drug_6.csv"
local_temp = "clean_drug_lean.csv"

anchor_df = pd.read_csv(anchor_path)

if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)

anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], format='ISO8601', utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], format='ISO8601', utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False

for chunk in pd.read_csv(source_csv, chunksize=chunk_size):
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
    chunk['drug_exposure_start_datetime'] = pd.to_datetime(
        chunk['drug_exposure_start_datetime'], format='ISO8601', utc=True
    ).dt.tz_localize(None)

    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    
    in_window = (chunk['drug_exposure_start_datetime'] >= chunk['win_start']) & \
                (chunk['drug_exposure_start_datetime'] < chunk['win_end'])
    
    chunk.loc[~in_window, 'drug_exposure_start_datetime'] = pd.NaT
    
    chunk['drug_exposure_start_datetime'] = chunk['drug_exposure_start_datetime'].dt.strftime('%Y-%m-%d')
    cols_to_save = [c for c in chunk.columns if c not in ['win_start', 'win_end']]
    chunk_to_save = chunk[cols_to_save]
    
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)

print(f"saved to: {target_csv}")

final_preview = pd.read_csv(target_csv, nrows=5)
display(final_preview.head(5))

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/surveys.csv" 
target_csv =  f"{bucket}/amia/clean_data/clean_survey_6.csv"
local_temp = "clean_survey_lean.csv"

anchor_df = pd.read_csv(anchor_path)

if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)
anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], format='ISO8601', utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], format='ISO8601', utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False

for chunk in pd.read_csv(source_csv, chunksize=chunk_size):
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
    chunk['survey_datetime'] = pd.to_datetime(
        chunk['survey_datetime'], format='ISO8601', utc=True
    ).dt.tz_localize(None)
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    in_window = (chunk['survey_datetime'] >= chunk['win_start']) & \
                (chunk['survey_datetime'] < chunk['win_end'])
    
    chunk.loc[~in_window, 'survey_datetime'] = pd.NaT
    chunk['survey_datetime'] = chunk['survey_datetime'].dt.strftime('%Y-%m-%d')
    cols_to_save = [c for c in chunk.columns if c not in ['win_start', 'win_end']]
    chunk_to_save = chunk[cols_to_save]
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

print(f"updating...")
subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)
print(f"saved: {target_csv}")

final_preview = pd.read_csv(target_csv, nrows=5)
display(final_preview.head(5))

In [ ]:
import pandas as pd
survey_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_survey_6.csv"

df_stats = pd.read_csv(survey_path, usecols=['survey_datetime'])
total_rows = len(df_stats)
na_count = df_stats['survey_datetime'].isna().sum()
na_percentage = (na_count / total_rows) * 100

print(f"total rows: {total_rows:,}")
print(f"NA time rows: {na_count:,}")
print(f"out of window ratio: {na_percentage:.2f}%")

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display
bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
anchor_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
source_csv = f"{bucket}/amia/observations.csv" 
target_csv = f"{bucket}/amia/clean_data/clean_observation_6.csv"
local_temp = "clean_observation_lean.csv"

anchor_df = pd.read_csv(anchor_path)
if 'patient_id' in anchor_df.columns:
    anchor_df.rename(columns={'patient_id': 'person_id'}, inplace=True)
anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start'], format='ISO8601', utc=True).dt.tz_localize(None)
anchor_df['end'] = pd.to_datetime(anchor_df['index_date'], format='ISO8601', utc=True).dt.tz_localize(None)

window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
allowed_ids = set(window_map.keys())

chunk_size = 200000
header_written = False

cols_to_keep = [
    'person_id', 
    'standard_concept_name', 
    'standard_concept_code', 
    'standard_vocabulary', 
    'observation_datetime', 
    'value_as_number', 'value_as_string', 'value_as_concept_name',
    'unit_concept_id', 
    'unit_concept_name'
]
for chunk in pd.read_csv(source_csv, usecols=cols_to_keep, chunksize=chunk_size):
    chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
    if chunk.empty:
        continue
    chunk['observation_datetime'] = pd.to_datetime(
        chunk['observation_datetime'], format='ISO8601', utc=True
    ).dt.tz_localize(None)
    chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
    chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
    in_window = (chunk['observation_datetime'] >= chunk['win_start']) & \
                (chunk['observation_datetime'] < chunk['win_end'])
    chunk.loc[~in_window, 'observation_datetime'] = pd.NaT
    chunk['observation_datetime'] = chunk['observation_datetime'].dt.strftime('%Y-%m-%d')
    chunk_to_save = chunk[cols_to_keep]
    mode = 'w' if not header_written else 'a'
    chunk_to_save.to_csv(local_temp, index=False, mode=mode, header=not header_written)
    header_written = True

subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
if os.path.exists(local_temp):
    os.remove(local_temp)

print(f"saved to: {target_csv}")

final_preview = pd.read_csv(target_csv, nrows=5)
display(final_preview.head(5))

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/

In [ ]:
!gsutil -m mv gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/clean_data gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/

# Timeframe=12 months

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display
bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
source_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
target_path = f"{bucket}/amia/clean_data/clean_positive_12.csv"
local_file = "clean_positive_12_fixed.csv"

df_raw = pd.read_csv(source_path)

print(f"total record before processing {len(df_raw)}")
print(f"independent patients number of record before processing: {df_raw['person_id'].nunique()}")
df_raw['index_date'] = pd.to_datetime(df_raw['index_date'], format='ISO8601', errors='coerce').dt.date
df_sorted = df_raw.sort_values(by=['person_id', 'index_date'], ascending=True)
df_unique = df_sorted.drop_duplicates(subset=['person_id'], keep='first').copy()
df_unique['timeframe_start'] = (pd.to_datetime(df_unique['index_date']) - pd.DateOffset(months=12)).dt.date
df_final = df_unique[['person_id', 'index_date', 'timeframe_start', 'IsPositive']]

df_final.to_csv(local_file, index=False)
subprocess.run(["gsutil", "cp", local_file, target_path], check=True)

print(f"final patients number: {len(df_final)}")

display(df_final.head(5))

In [ ]:
def process_omop_table_final(table_name, date_col, anchor_path, target_name, cols_to_keep=None):
    source_csv = f"{bucket}/amia/{table_name}"
    target_csv = f"{bucket}/amia/clean_data/{target_name}"
    local_temp = f"temp_{target_name}"
    anchor_df = pd.read_csv(anchor_path)
    anchor_df['start'] = pd.to_datetime(anchor_df['timeframe_start']).dt.tz_localize(None)
    anchor_df['end'] = pd.to_datetime(anchor_df['index_date']).dt.tz_localize(None)
    window_map = anchor_df.set_index('person_id')[['start', 'end']].to_dict('index')
    allowed_ids = set(window_map.keys())

    chunk_size = 500000
    header_written = False
    total_rows = 0
    null_rows = 0

    print(f"processing {table_name}...")
    read_params = {"chunksize": chunk_size}
    if cols_to_keep: read_params["usecols"] = cols_to_keep
    for chunk in pd.read_csv(source_csv, **read_params):
        chunk = chunk[chunk['person_id'].isin(allowed_ids)].copy()
        if chunk.empty: continue
        chunk[date_col] = pd.to_datetime(chunk[date_col], format='ISO8601', utc=True).dt.tz_localize(None)
        chunk['win_start'] = chunk['person_id'].map(lambda x: window_map[x]['start'])
        chunk['win_end'] = chunk['person_id'].map(lambda x: window_map[x]['end'])
        in_window = (chunk[date_col] >= chunk['win_start']) & (chunk[date_col] < chunk['win_end'])
        chunk.loc[~in_window, date_col] = pd.NaT
        total_rows += len(chunk)
        null_rows += chunk[date_col].isna().sum()
        chunk[date_col] = chunk[date_col].dt.strftime('%Y-%m-%d')
        final_cols = [c for c in chunk.columns if c not in ['win_start', 'win_end']]
        
        mode = 'w' if not header_written else 'a'
        chunk[final_cols].to_csv(local_temp, index=False, mode=mode, header=not header_written)
        header_written = True

    subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
    if os.path.exists(local_temp): os.remove(local_temp)
        
    null_rate = (null_rows / total_rows * 100) if total_rows > 0 else 0
    print(f"({target_name}):")
    print(f" total records number{total_rows}")
    print(f" out of timeframe{null_rows}")
    print(f" data loss {null_rate:.2f}%")
    
    display(pd.read_csv(target_csv, nrows=5))

In [ ]:
process_omop_table_final(
    table_name='measurements.csv', 
    date_col='measurement_datetime', 
    anchor_path=target_path, 
    target_name='clean_measurement_12.csv',
   cols_to_keep = [
    'person_id', 'standard_concept_name', 'standard_concept_code', 
    'standard_vocabulary', 'measurement_datetime', 
    'value_as_number', 'value_as_concept_id','value_as_concept_name','unit_concept_id', 'unit_concept_name'
]
)

In [ ]:
# Observation
process_omop_table_final(
    table_name='observations.csv', 
    date_col='observation_datetime', 
    anchor_path=target_path, 
    target_name='clean_observation_12.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'observation_datetime', 'value_as_number', 'value_as_string', 'value_as_concept_name', 'unit_concept_id', 'unit_concept_name']
)

In [ ]:
#Condition 
process_omop_table_final(
    table_name='conditions.csv', 
    date_col='condition_start_datetime', 
    anchor_path=target_path, 
    target_name='clean_condition_12.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 'condition_start_datetime']
)

# Drug
process_omop_table_final(
    table_name='drugs.csv', 
    date_col='drug_exposure_start_datetime', 
    anchor_path=target_path, 
    target_name='clean_drug_12.csv'
)

# Lab
process_omop_table_final(
    table_name='labs.csv', 
    date_col='measurement_datetime', 
    anchor_path=target_path, 
    target_name='clean_lab_12.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'measurement_datetime', 'value_as_number', 'value_as_concept_id', 'value_as_concept_name', 
                  'unit_concept_id', 'unit_concept_name', 'range_low', 'range_high']
)



# Procedure
process_omop_table_final(
    table_name='procedures.csv', 
    date_col='procedure_datetime', 
    anchor_path=target_path, 
    target_name='clean_procedure_12.csv',
    cols_to_keep=['person_id','standard_concept_name','standard_concept_code','standard_vocabulary','procedure_datetime']
)

# Survey
process_omop_table_final(
    table_name='surveys.csv', 
    date_col='survey_datetime', 
    anchor_path=target_path, 
    target_name='clean_survey_12.csv',
    cols_to_keep=['person_id','survey_datetime','survey','question_concept_id','question','answer_concept_id','answer']
)

print(f"saved to: {bucket}/amia/clean_data/")

In [ ]:
process_omop_table_final(
    table_name='surveys.csv', 
    date_col='survey_datetime', 
    anchor_path=target_path, 
    target_name='clean_survey_12.csv',
    cols_to_keep=['person_id','survey_datetime','survey','question_concept_id','question','answer_concept_id','answer']
)

# TimeFrame=24 months

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display
bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
source_path = f"{bucket}/amia/clean_data/clean_positive_6.csv"
target_path = f"{bucket}/amia/clean_data/clean_positive_24.csv"
local_file = "clean_positive_12_fixed.csv"

df_raw = pd.read_csv(source_path)
print(f"total records before processing: {len(df_raw)}")
print(f"independent patients number before preprocessing: {df_raw['person_id'].nunique()}")
df_raw['index_date'] = pd.to_datetime(df_raw['index_date'], format='ISO8601', errors='coerce').dt.date
df_sorted = df_raw.sort_values(by=['person_id', 'index_date'], ascending=True)
df_unique = df_sorted.drop_duplicates(subset=['person_id'], keep='first').copy()
df_unique['timeframe_start'] = (pd.to_datetime(df_unique['index_date']) - pd.DateOffset(months=24)).dt.date
df_final = df_unique[['person_id', 'index_date', 'timeframe_start', 'IsPositive']]

df_final.to_csv(local_file, index=False)
subprocess.run(["gsutil", "cp", local_file, target_path], check=True)

print(f"final patients number: {len(df_final)}")
display(df_final.head(5))

In [ ]:
# Observation
process_omop_table_final(
    table_name='observations.csv', 
    date_col='observation_datetime', 
    anchor_path=target_path, 
    target_name='clean_observation_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'observation_datetime',  'value_as_number', 'value_as_string', 'value_as_concept_name', 'unit_concept_id', 'unit_concept_name']
)

In [ ]:
# Measurement
process_omop_table_final(
    table_name='measurements.csv', 
    date_col='measurement_datetime', 
    anchor_path=target_path, 
    target_name='clean_measurement_24.csv',
    cols_to_keep = [
    'person_id', 'standard_concept_name', 'standard_concept_code', 
    'standard_vocabulary', 'measurement_datetime', 
    'value_as_number', 'value_as_concept_id','value_as_concept_name','unit_concept_id', 'unit_concept_name'
]
)

In [ ]:
# Condition
process_omop_table_final(
    table_name='conditions.csv', 
    date_col='condition_start_datetime', 
    anchor_path=target_path, 
    target_name='clean_condition_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 'condition_start_datetime']
)

# Drug
process_omop_table_final(
    table_name='drugs.csv', 
    date_col='drug_exposure_start_datetime', 
    anchor_path=target_path, 
    target_name='clean_drug_24.csv'
)

# Lab
process_omop_table_final(
    table_name='labs.csv', 
    date_col='measurement_datetime', 
    anchor_path=target_path, 
    target_name='clean_lab_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'measurement_datetime', 'value_as_number', 'value_as_concept_id', 'value_as_concept_name', 
                  'unit_concept_id', 'unit_concept_name', 'range_low', 'range_high']
)

# Procedure
process_omop_table_final(
    table_name='procedures.csv', 
    date_col='procedure_datetime', 
    anchor_path=target_path, 
    target_name='clean_procedure_24.csv',
    cols_to_keep=['person_id','standard_concept_name','standard_concept_code','standard_vocabulary','procedure_datetime']
)

# Survey
process_omop_table_final(
    table_name='surveys.csv', 
    date_col='survey_datetime', 
    anchor_path=target_path, 
    target_name='clean_survey_24.csv',
    cols_to_keep=['person_id','survey_datetime','survey','question_concept_id','question','answer_concept_id','answer']
)

print(f"saved to: {bucket}/amia/clean_data/")

In [ ]:
!gsutil ls gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/clean_data/

# Negative

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display
bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
raw_dir = f"{bucket}/amia"
clean_dir = f"{bucket}/amia/clean_data"
positive_anchor_path = f"{clean_dir}/clean_positive_6.csv"

tables_config = [
    {'name': 'conditions.csv', 'date_col': 'condition_start_datetime', 'short': 'condition'},
    {'name': 'drugs.csv',      'date_col': 'drug_exposure_start_datetime', 'short': 'drug'},
    {'name': 'labs.csv',       'date_col': 'measurement_datetime', 'short': 'lab'},
    {'name': 'measurements.csv','date_col': 'measurement_datetime', 'short': 'measurement'},
    {'name': 'observations.csv','date_col': 'observation_datetime', 'short': 'observation'},
    {'name': 'procedures.csv',  'date_col': 'procedure_datetime', 'short': 'procedure'},
    {'name': 'surveys.csv',     'date_col': 'survey_datetime', 'short': 'survey'}
]

In [ ]:
def build_negative_anchor():
    pos_df = pd.read_csv(positive_anchor_path)
    pos_ids = set(pos_df['person_id'].unique())
    negative_max_dates = {}
    for config in tables_config:
        file_path = f"{raw_dir}/{config['name']}"
        date_col = config['date_col']
        print(f"preprocessing: {config['name']}")
        try:
            for chunk in pd.read_csv(file_path, usecols=['person_id', date_col], chunksize=500000):
                chunk = chunk[~chunk['person_id'].isin(pos_ids)].copy()
                if chunk.empty: continue
                chunk[date_col] = pd.to_datetime(chunk[date_col], format='ISO8601', utc=True, errors='coerce').dt.tz_localize(None)
                chunk_max = chunk.groupby('person_id')[date_col].max()
                for pid, m_date in chunk_max.items():
                    if pd.isna(m_date): continue
                    if pid not in negative_max_dates or m_date > negative_max_dates[pid]:
                        negative_max_dates[pid] = m_date
        except Exception as e:
            print(f"skip {config['name']} or error: {e}")

    neg_anchor = pd.DataFrame(list(negative_max_dates.items()), columns=['person_id', 'index_date'])
    neg_anchor['timeframe_start'] = neg_anchor['index_date'] - pd.DateOffset(months=6)
    neg_anchor['IsPositive'] = 0
    neg_anchor['index_date'] = neg_anchor['index_date'].dt.strftime('%Y-%m-%d')
    neg_anchor['timeframe_start'] = neg_anchor['timeframe_start'].dt.strftime('%Y-%m-%d')
    neg_anchor_path = f"{clean_dir}/clean_negative_anchor_6.csv"
    neg_anchor.to_csv("temp_neg_anchor.csv", index=False)
    subprocess.run(["gsutil", "cp", "temp_neg_anchor.csv", neg_anchor_path], check=True)
    os.remove("temp_neg_anchor.csv")
    
    print(f"There are {len(neg_anchor)} negative patients")
    display(neg_anchor.head())
    return neg_anchor_path
negative_anchor_file = build_negative_anchor()

In [ ]:
import pandas as pd
from tqdm import tqdm

neg_anchor_path = f"{clean_dir}/clean_negative_anchor_6.csv"
print(f"Scan: {neg_anchor_path}")
df = pd.read_csv(neg_anchor_path)
df['index_date'] = pd.to_datetime(df['index_date'])
earliest_date = df['index_date'].min()
latest_date = df['index_date'].max()
print(f" total number: {len(df):,}")
print(f"earlist Index Date: {earliest_date.strftime('%Y-%m-%d')}")
print(f"latest Index Date: {latest_date.strftime('%Y-%m-%d')}")
target_date_str = '2020-01-01' 
def check_date_threshold(date_str):
    threshold = pd.to_datetime(date_str)
    filtered_df = df[df['index_date'] > threshold]
    print(f"date > {date_str}):")
    print(f"remain patients number {len(filtered_df):,}")
    print(f"total number of people ration:{len(filtered_df)/len(df):.2%}")
    
    if len(filtered_df) > 0:
        print(f"earlist date in range: {filtered_df['index_date'].min().strftime('%Y-%m-%d')}")
        print(f"latest date in range: {filtered_df['index_date'].max().strftime('%Y-%m-%d')}")
    return filtered_df

subset_df = check_date_threshold(target_date_str)

In [ ]:
import pandas as pd
import os
import subprocess
from IPython.display import display
from tqdm.auto import tqdm 

def process_negative_table_final(table_name, date_col, target_name, cols_to_keep=None):
    source_csv = f"{raw_dir}/{table_name}"
    target_csv = f"{clean_dir}/{target_name}"
    local_temp = f"temp_{target_name}"
    anchor_path = f"{clean_dir}/clean_negative_anchor_24.csv"
    anchor_df = pd.read_csv(anchor_path)
    anchor_df['start_dt'] = pd.to_datetime(anchor_df['timeframe_start']).dt.tz_localize(None)
    anchor_df['end_dt'] = pd.to_datetime(anchor_df['index_date']).dt.tz_localize(None)
    neg_window_map = anchor_df.set_index('person_id')[['start_dt', 'end_dt']].to_dict('index')
    neg_allowed_ids = set(neg_window_map.keys())

    chunk_size = 500000
    header_written = False
    total_rows, null_rows = 0, 0

    print(f"[Processing] {table_name} -> {target_name}")
    try:
        file_stats = subprocess.check_output(["gsutil", "du", source_csv]).decode('utf-8')
        file_size = int(file_stats.split()[0])
    except:
        file_size = None
    dtype_dict = {
        'standard_concept_code': str,
        'standard_concept_name': str,
        'standard_vocabulary': str,
        'person_id': int 
    }

    read_params = {
        "chunksize": chunk_size,
        "dtype": dtype_dict,
        "low_memory": False
    }
    if cols_to_keep: 
        read_params["usecols"] = cols_to_keep
    pbar = tqdm(pd.read_csv(source_csv, **read_params), desc=f"Reading {table_name}")

    for chunk in pbar:
        chunk = chunk[chunk['person_id'].isin(neg_allowed_ids)].copy()
        if chunk.empty: 
            continue
        chunk[date_col] = pd.to_datetime(chunk[date_col], format='ISO8601', utc=True, errors='coerce').dt.tz_localize(None)
        chunk['win_start'] = chunk['person_id'].map(lambda x: neg_window_map[x]['start_dt'])
        chunk['win_end'] = chunk['person_id'].map(lambda x: neg_window_map[x]['end_dt'])   
        in_window = (chunk[date_col] >= chunk['win_start']) & (chunk[date_col] < chunk['win_end'])       
        total_rows += len(chunk)
        null_rows += (~in_window).sum()
        chunk.loc[~in_window, date_col] = pd.NaT
        chunk[date_col] = chunk[date_col].dt.strftime('%Y-%m-%d')
        final_cols = [c for c in chunk.columns if c not in ['win_start', 'win_end']]
        mode = 'w' if not header_written else 'a'
        chunk[final_cols].to_csv(local_temp, index=False, mode=mode, header=not header_written)
        header_written = True
        current_loss = (null_rows / total_rows * 100) if total_rows > 0 else 0
        pbar.set_postfix({"loss": f"{current_loss:.1f}%"})

    subprocess.run(["gsutil", "cp", local_temp, target_csv], check=True)
    if os.path.exists(local_temp): 
        os.remove(local_temp)
    
    loss_rate = (null_rows / total_rows * 100) if total_rows > 0 else 0
    print(f" {target_name} date loss: {loss_rate:.2f}%")
    display(pd.read_csv(target_csv, nrows=5))

In [ ]:
# Condition
process_negative_table_final(
    table_name='conditions.csv',
    date_col='condition_start_datetime',
    target_name='clean_negative_condition_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 'condition_start_datetime']
)

In [ ]:
# Drug
process_negative_table_final(
    table_name='drugs.csv',
    date_col='drug_exposure_start_datetime',
    target_name='clean_negative_drug_24.csv',
    
)

In [ ]:
# Lab
process_negative_table_final(
    table_name='labs.csv',
    date_col='measurement_datetime',
    target_name='clean_negative_lab_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'measurement_datetime', 'value_as_number', 'value_as_concept_id', 'value_as_concept_name', 
                  'unit_concept_id', 'unit_concept_name', 'range_low', 'range_high']
)

In [ ]:
# Measurement
process_negative_table_final(
    table_name='measurements.csv',
    date_col='measurement_datetime',
    target_name='clean_negative_measurement_24.csv',
    cols_to_keep = [
    'person_id', 'standard_concept_name', 'standard_concept_code', 
    'standard_vocabulary', 'measurement_datetime', 
    'value_as_number', 'value_as_concept_id','value_as_concept_name','unit_concept_id', 'unit_concept_name'
    ]
        )

In [ ]:
#Observation
process_negative_table_final(
    table_name='observations.csv',
    date_col='observation_datetime',
    target_name='clean_negative_observation_24.csv',
        cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 
                  'observation_datetime',  'value_as_number', 'value_as_string', 'value_as_concept_name', 'unit_concept_id', 'unit_concept_name']
)

In [ ]:
# Procedure
process_negative_table_final(
    table_name='procedures.csv',
    date_col='procedure_datetime',
    target_name='clean_negative_procedure_24.csv',
    cols_to_keep=['person_id', 'standard_concept_name', 'standard_concept_code', 'standard_vocabulary', 'procedure_datetime']
)

# Survey
process_negative_table_final(
    table_name='surveys.csv',
    date_col='survey_datetime',
    target_name='clean_negative_survey_24.csv',
    cols_to_keep=['person_id','survey_datetime','survey','question_concept_id','question','answer_concept_id','answer']
)


In [ ]:
import subprocess

bucket = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf"
subprocess.run(["gsutil", "ls", "-lh", f"{bucket}/amia/clean_data/"])

# 12 months negative

In [ ]:
import pandas as pd
from tqdm import tqdm
import subprocess
import os
source_gs_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_negative_anchor_6.csv"
target_gs_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_negative_anchor_12.csv"
local_temp_file = "temp_neg_anchor.csv"
subprocess.run(["gsutil", "cp", source_gs_path, local_temp_file], check=True)

df = pd.read_csv(local_temp_file)
print(f"{len(df):,} rows in total")
df['index_date'] = pd.to_datetime(df['index_date'])
df['index_date'] = df['index_date'].dt.strftime('%Y-%m-%d')
df['timeframe_start'] = df['timeframe_start'].dt.strftime('%Y-%m-%d')

output_local = "clean_negative_anchor_12.csv"
df.to_csv(output_local, index=False)
subprocess.run(["gsutil", "cp", output_local, target_gs_path], check=True)
if os.path.exists(local_temp_file): os.remove(local_temp_file)
if os.path.exists(output_local): os.remove(output_local)

print("done")

# 24 months negative

In [ ]:
import pandas as pd
from tqdm import tqdm
import subprocess
import os
source_gs_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_negative_anchor_6.csv"
target_gs_path = "gs://fc-secure-45314987-e7dc-4f2a-888d-f18f606c1cbf/amia/clean_data/clean_negative_anchor_24.csv"
local_temp_file = "temp_neg_anchor.csv"

subprocess.run(["gsutil", "cp", source_gs_path, local_temp_file], check=True)
df = pd.read_csv(local_temp_file)
print(f"{len(df):,}rows in total")
df['index_date'] = pd.to_datetime(df['index_date'])
df['timeframe_start'] = df['index_date'] - pd.DateOffset(months=24)
df['index_date'] = df['index_date'].dt.strftime('%Y-%m-%d')
df['timeframe_start'] = df['timeframe_start'].dt.strftime('%Y-%m-%d')

output_local = "clean_negative_anchor_24.csv"
df.to_csv(output_local, index=False)
subprocess.run(["gsutil", "cp", output_local, target_gs_path], check=True)

if os.path.exists(local_temp_file): os.remove(local_temp_file)
if os.path.exists(output_local): os.remove(output_local)
print(" done")